In [1]:
%pip install datasets
%pip install torchvision
%pip install pillow
!pip install --upgrade datasets
%pip install opencv-python
%pip install tensorboard


from datasets import load_dataset

ds = load_dataset("phunc20/nj_biergarten_captcha")
from PIL import Image
print("Pillow version:", Image.__version__)



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Pillow version: 11.2.1


In [2]:
from PIL import Image
print("Pillow version:", Image.__version__)


Pillow version: 11.2.1


In [3]:

# Print the key of the first example in the training set
print(ds["train"][0]["__key__"])
#Show the image
ds["train"][0]["jpg"].show()
# Extract the label by extracting characters after '_'
print(*ds["train"][1]["__key__"].split("_")[1:])
# Print size of dataset
print(len(ds["train"]))
# Print shape of image
print(ds["train"][0]["jpg"].size)

001/2024-10-08T18:21:10.578994_y2jhuv
34veh5
476118
(140, 50)


In [4]:
import numpy as np
import torch
import cv2

from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
import matplotlib.pyplot as plt

import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.transforms import functional as F

from torch.optim import Adam
from tqdm import tqdm

from torchvision import transforms
from datasets import load_dataset
from PIL import Image

from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter

#PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
torch.cuda.empty_cache()

print(torch.cuda.is_available())  # Should return True
print(torch.cuda.get_device_name(0))  # Displays your GPU name

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

True
Tesla T4
Using device: cuda


* Functions

In [5]:


def char_to_1_hot(char, sorted_chars):
    # Convert a character to a one-hot encoded vector
    index = sorted_chars.index(char)
    one_hot = np.zeros(len(sorted_chars), dtype=np.uint8)
    one_hot[index] = 1.0
    return one_hot

def one_hot(label, sorted_chars):
    return np.hstack([char_to_1_hot(char, sorted_chars) for char in label[0]])

def one_hot_to_char(x, sorted_chars):
    y = np.array(x)
    y = y.squeeze()
    assert len(y) == len(sorted_chars)
    idx = np.argmax(y)
    return(sorted_chars[idx])

def one_hot_to_label(x, sorted_chars, char_per_label):
    y = np.array(x)
    y = y.squeeze()
    label_list = []
    assert len(y) == (len(sorted_chars * char_per_label))
    for i in range(0, char_per_label):
        start = i * len(sorted_chars)
        end = start + len(sorted_chars)
        label_list.append(one_hot_to_char(y[start:end], sorted_chars))
    return "".join(label_list)



In [6]:
def extract_labels(dataset):
    # Extract the labels from the dataset
    val = np.array(dataset.split("_")[1:])
    return val
    #*ds["train"][0]["__key__"].split("_")[1:]
class CustomCaptchaDataset(Dataset):
    def __init__(self, transform=None, sorted_chars=None):
        self.transform = transform
        self.sorted_chars = sorted_chars

    def __len__(self):
        return len(ds["train"])

    def __getitem__(self, idx):

        label = one_hot(extract_labels(ds["train"][idx]["__key__"]), self.sorted_chars)
        image = np.array(ds["train"][idx]["jpg"])

        if self.transform:
            image = self.transform(image)

        return image, label

    def get_labels(self, range):
        label = extract_labels(ds["train"][range]["__key__"])
        return label

In [7]:
data_points = 476118
batch_size = 128
char_per_label = 6

sorted_chars = ['1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

train_size = int(data_points * 0.75)
test_size = int(data_points - train_size)
print(f"Train size: {train_size}, Test size: {test_size}")

# Grayscale the data
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Resize((25, 70)),
    transforms.GaussianBlur(kernel_size=(3, 3), sigma=0.1)
])

dataset = CustomCaptchaDataset(transform=transform, sorted_chars=sorted_chars)

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# Create DataLoaders for batch processing\n",
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(dataset[0][0].shape)
print(dataset[0][1])

Train size: 357088, Test size: 119030
torch.Size([1, 25, 70])
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0]


In [8]:
# Get all of the characters in the dataset
sorted_chars = ['1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
test_string = "34veh5"

print(len(sorted_chars))
print(one_hot(test_string, sorted_chars))
print(one_hot_to_label(dataset[1][1], sorted_chars, char_per_label))

35
[0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
34veh5


In [9]:
class FasterRCNNDataset(Dataset):
    def __init__(self, dataset, sorted_chars, transform=None):
        self.dataset = dataset
        self.sorted_chars = sorted_chars
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # Retrieve image and label
        image, label = self.dataset[idx]
        label = one_hot_to_label(label, self.sorted_chars, char_per_label)

        # Generate bounding boxes and labels
        target = {
            'boxes': torch.tensor(self.generate_boxes(label), dtype=torch.float32),
            'labels': torch.tensor(self.generate_labels(label), dtype=torch.int64),
        }

        # Apply transformations
        if self.transform:
            image = self.transform(image.numpy())

        # Convert image to 3-channel format
        image = torch.tensor(image, dtype=torch.float32).repeat(3, 1, 1)

        return image, target

    def generate_boxes(self, label):
        # Generate bounding boxes for each character
        num_chars = len(label)
        char_width = 50 / num_chars
        boxes = []
        for i in range(num_chars):
            x_min = i * char_width + 7
            x_max = x_min + char_width
            y_min = 2
            y_max = 20
            boxes.append([x_min, y_min, x_max, y_max])
        return boxes

    def generate_labels(self, label):
        return [self.sorted_chars.index(c) + 1 for c in label]




torch.cuda.empty_cache()

In [12]:
# Initialize Faster R-CNN model without pretrained weights
num_classes = len(sorted_chars) + 1
model = fasterrcnn_resnet50_fpn(pretrained=False)

# Replace the ROI head
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)

# Optional: Initialize backbone weights
def initialize_weights(module):
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
        if module.bias is not None:
            nn.init.constant_(module.bias, 0)

model.backbone.apply(initialize_weights)
model.to(device)

def collate_fn(batch):
    images, targets = zip(*batch)  # Unzip the batch
    return list(images), list(targets)


# Define optimizer
optimizer = Adam(model.parameters(), lr=0.0001)

# Prepare datasets and dataloaders
dataset_train = FasterRCNNDataset(train_dataset, sorted_chars)
dataset_test = FasterRCNNDataset(test_dataset, sorted_chars)

train_loader = DataLoader(dataset_train, batch_size=128, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(dataset_test, batch_size=128, shuffle=False, collate_fn=collate_fn)


# Initialize metrics and TensorBoard writer
epoch_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []
writer = SummaryWriter()

best_val_loss = float('inf')


# Training and evaluation loop
epochs = 20
for epoch in range(epochs):
    # Training phase
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
    for images, targets in progress_bar:
        # Prepare inputs and targets
        images = [image.to(device) for image in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        try:
            # Forward pass
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            # Backward pass and optimizer step
            losses.backward()
            optimizer.step()

            total_loss += losses.item()
        except Exception as e:
            print(f"Error during training: {e}")
            continue

        # Update progress bar
        progress_bar.set_postfix(loss=f"{total_loss / len(progress_bar):.4f}")

    avg_train_loss = total_loss / len(train_loader)
    epoch_losses.append(avg_train_loss)
    writer.add_scalar('Train Loss', avg_train_loss, epoch)

    # Validation phase
    model.eval()
    val_loss = 0
    progress_bar = tqdm(test_loader, desc=f"Epoch {epoch+1}/{epochs} [Valid]")
    with torch.no_grad():
        for images, targets in progress_bar:
            # Prepare inputs and targets
            images = [image.to(device) for image in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            try:
                # Forward pass
                loss_dict = model(images, targets)
                losses = sum(loss for loss in loss_dict.values())
                val_loss += losses.item()
            except Exception as e:
                print(f"Error during validation: {e}")
                continue

            # Update progress bar
            progress_bar.set_postfix(loss=f"{val_loss / len(progress_bar):.4f}")

    avg_val_loss = val_loss / len(test_loader)
    test_losses.append(avg_val_loss)
    writer.add_scalar('Validation Loss', avg_val_loss, epoch)

    # Save best model based on validation loss
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"Model saved at epoch {epoch+1} with validation loss: {avg_val_loss:.4f}")

    # Epoch summary
    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

# Plot metrics after training
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, epochs+1), epoch_losses, label='Train Loss')
plt.plot(range(1, epochs+1), test_losses, label='Validation Loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
# Placeholder for train/validation accuracy, since this example doesn't compute accuracy
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()

plt.tight_layout()
plt.show()

print("Training complete.")

Epoch 1/20 [Train]:   0%|          | 0/2790 [00:00<?, ?it/s]<ipython-input-9-66296486fe83>:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  image = torch.tensor(image, dtype=torch.float32).repeat(3, 1, 1)
Epoch 1/20 [Train]:   0%|          | 1/2790 [00:00<32:17,  1.44it/s]

Error during training: CUDA out of memory. Tried to allocate 4.92 GiB. GPU 0 has a total capacity of 14.74 GiB of which 444.12 MiB is free. Process 481381 has 14.30 GiB memory in use. Of the allocated memory 11.25 GiB is allocated by PyTorch, and 2.92 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


Epoch 1/20 [Train]:   0%|          | 2/2790 [00:01<22:21,  2.08it/s]

Error during training: CUDA out of memory. Tried to allocate 4.92 GiB. GPU 0 has a total capacity of 14.74 GiB of which 444.12 MiB is free. Process 481381 has 14.30 GiB memory in use. Of the allocated memory 11.26 GiB is allocated by PyTorch, and 2.91 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


Epoch 1/20 [Train]:   0%|          | 3/2790 [00:01<18:49,  2.47it/s]

...

Epoch 1/20 [Train]:   4%|▎         | 102/2790 [00:30<12:49,  3.49it/s]

Error during training: CUDA out of memory. Tried to allocate 4.92 GiB. GPU 0 has a total capacity of 14.74 GiB of which 444.12 MiB is free. Process 481381 has 14.30 GiB memory in use. Of the allocated memory 11.26 GiB is allocated by PyTorch, and 2.91 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


Epoch 1/20 [Train]:   4%|▎         | 103/2790 [00:30<13:24,  3.34it/s]

Error during training: CUDA out of memory. Tried to allocate 4.92 GiB. GPU 0 has a total capacity of 14.74 GiB of which 444.12 MiB is free. Process 481381 has 14.30 GiB memory in use. Of the allocated memory 11.25 GiB is allocated by PyTorch, and 2.92 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


KeyboardInterrupt: 